# Image Restoration Interactive Demo
Run the cells below to initialize the pipeline, upload a degraded image (haze, low-light, rain), and view the automatic restoration results.

In [ ]:
import sys
import os
from pathlib import Path

# Add the project root to the Python path so we can import our modules
project_root = str(Path(os.getcwd()).parent)
if project_root not in sys.path:
    sys.path.append(project_root)

import io
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output

from src.core.pipeline import RestorationPipeline
from src.utils.image import save_image


## 1. Initialize Pipeline
This will load the configuration and prepare the degradation router and metrics engines. Models are loaded lazily upon first use.

In [ ]:
print("Initializing pipeline...")
pipeline = RestorationPipeline()
print(f"Pipeline initialized on device: {pipeline.config.pipeline.device}")

## 2. Interactive Uploader
Run the cell below to display the upload button. Upload an image, and the system will automatically analyze and restore it.

In [ ]:
upload_widget = widgets.FileUpload(accept='image/*', multiple=False)
output_ui = widgets.Output()

def process_uploaded_image(change):
    with output_ui:
        clear_output(wait=True)
        
        # Handle both old and new ipywidgets FileUpload dictionary structures
        uploaded_file = upload_widget.value
        if isinstance(uploaded_file, tuple):
            # New ipywidgets >= 8.0 format
            content = uploaded_file[0]['content']
            filename = uploaded_file[0]['name']
        else:
            # Old ipywidgets < 8.0 format
            filename = list(uploaded_file.keys())[0]
            content = uploaded_file[filename]['content']
            
        print(f"\nProcessing {filename}...")
        
        # Save temp file for pipeline to process
        temp_input = Path("temp_input.jpg")
        temp_output = Path("temp_output.jpg")
        
        with open(temp_input, "wb") as f:
            f.write(content)
            
        # Run pipeline
        try:
            results = pipeline.process_image(temp_input, temp_output)
            
            print("\n--- Analysis Complete ---")
            print(f"Detected Scores: {results['scores']}")
            print(f"Execution Plan : {results['plan']}")
            print(f"Latency        : {results['latency_ms']:.2f} ms")
            print(f"Quality Metrics: {results['metrics']}")
            
            # Display results side-by-side
            img_orig = Image.open(temp_input)
            
            fig, axes = plt.subplots(1, 2, figsize=(15, 8))
            axes[0].imshow(img_orig)
            axes[0].set_title("Original Image")
            axes[0].axis('off')
            
            if temp_output.exists() and results['plan']:
                img_restored = Image.open(temp_output)
                axes[1].imshow(img_restored)
                axes[1].set_title("Restored Image")
            else:
                axes[1].imshow(img_orig)
                axes[1].set_title("No Restoration Needed (Scores below threshold)")
                
            axes[1].axis('off')
            plt.tight_layout()
            plt.show()
            
            # Cleanup temp files
            if temp_input.exists(): temp_input.unlink()
            if temp_output.exists(): temp_output.unlink()
            
        except Exception as e:
            print(f"\nError during processing: {e}")

upload_widget.observe(process_uploaded_image, names='value')

display(widgets.VBox([widgets.Label("Upload an image to restore:"), upload_widget, output_ui]))